## Actividad: distribución de ganancias según hiperparámetros y datos de entrada

Basado en la sección "Tarea" de `monday/z201_Sobre_Árboles.ipynb` (commit
`9441950`, material de cátedra DMEyF 2026). Esta copia agrega el setup
necesario (imports, constantes de negocio y carga de datos con pandas) para
poder ejecutarse de forma independiente, y repite el entrenamiento con
`StratifiedShuffleSplit` para poder ver la **distribución** de la ganancia
en test (no solo un valor puntual) al variar los hiperparámetros del árbol
o el conjunto de datos de entrada (`mes_train`).

In [ ]:
import sys

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.tree import DecisionTreeClassifier

sys.path.insert(0, "..")
from config.semillas import SEMILLAS

DATA_PATH = "../datasets/processed/competencia_01.csv"

ganancia_acierto = 1_072_500
costo_estimulo = 27_500

# Carga con pandas: el dataset ya trae `clase_ternaria` calculada.
data = pd.read_csv(DATA_PATH)
print(f"filas: {data.shape[0]:,}  columnas: {data.shape[1]}")

In [ ]:
mes_train = 202103
mes_test = 202105

X = data[data['foto_mes'] == mes_train]
y = X['clase_ternaria']
X = X.drop(columns=['clase_ternaria'])


In [ ]:
import requests

url = "https://www.random.org/integers/"
params = {
    "num": 100,
    "min": 1,
    "max": 1_000_000_000,
    "col": 1,
    "base": 10,
    "format": "plain",
    "rnd": "new"
}

try:
    response = requests.get(url, params=params, timeout=5)
    response.raise_for_status()
    random_numbers_str = response.text.strip().split()
    semillas = [int(r) for r in random_numbers_str]
except requests.exceptions.RequestException as e:
    print(f"Ups: {e} — uso las semillas de config/semillas.py")
    semillas = SEMILLAS
print(semillas[:5])

#copie y pegelas en el algun lugar, no se van a repetir estos valores en su finita existencia

In [ ]:
def ganancia(model, X, y, prop=1, threshold=0.025):

  class_index = np.where(model.classes_ == "BAJA+2")[0][0]
  y_hat = model.predict_proba(X)

  @np.vectorize
  def ganancia_row(predicted, actual, threshold=0.025):
    return  (predicted >= threshold) * (ganancia_acierto if actual == "BAJA+2" else -costo_estimulo)

  return ganancia_row(y_hat[:,class_index], y).sum() / prop


In [ ]:
tree_params = {
    'criterion': 'gini',
    'max_depth': 5,
    'min_samples_split': 80
}

#lista de ganancias
gan_test = []
gan_train = [] # esta no se mira NUNCA, esta para sumar dibujitos

sss = StratifiedShuffleSplit(n_splits=10, test_size=0.3, random_state=semillas[0])

for train_index, test_index in sss.split(X, y):

  model = DecisionTreeClassifier(**tree_params)
  model.fit(X.iloc[train_index], y.iloc[train_index])

  gan_test.append(ganancia(model, X.iloc[test_index], y.iloc[test_index], prop=0.3))
  gan_train.append(ganancia(model, X.iloc[train_index], y.iloc[train_index], prop=0.7))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# DataFrame for plotting
df_gains = pd.DataFrame({
    'Ganancia': gan_test + gan_train,
    'Conjunto': ['Test'] * len(gan_test) + ['Train'] * len(gan_train)
})

# Plot histograms using FacetGrid to stack them vertically
g = sns.FacetGrid(df_gains, row='Conjunto', aspect=2, height=4)
g.map(sns.histplot, 'Ganancia', kde=True, palette='viridis')
g.set_axis_labels('Ganancia', 'Frecuencia')
g.set_titles(row_template='Conjunto: {row_name}')
plt.suptitle('Distribución de Ganancias por Conjunto', y=1.02) # Add a main title
plt.tight_layout() # Adjust layout to prevent overlapping titles/labels
plt.show()

## Comparación entre 2 configuraciones de hiperparámetros

Repetimos el mismo experimento (10 particiones con `StratifiedShuffleSplit`,
misma semilla) para dos árboles con hiperparámetros distintos, y comparamos
la distribución de ganancia en test entre ambos.

In [ ]:
def experimento_ganancia(tree_params, X, y, semilla, n_splits=10, test_size=0.3):
    gan_test = []
    sss = StratifiedShuffleSplit(n_splits=n_splits, test_size=test_size, random_state=semilla)

    for train_index, test_index in sss.split(X, y):
        model = DecisionTreeClassifier(**tree_params)
        model.fit(X.iloc[train_index], y.iloc[train_index])
        gan_test.append(ganancia(model, X.iloc[test_index], y.iloc[test_index], prop=test_size))

    return gan_test


tree_params_A = {
    'criterion': 'gini',
    'max_depth': 5,
    'min_samples_split': 80
}

tree_params_B = {
    'criterion': 'entropy',
    'max_depth': 10,
    'min_samples_split': 20
}

gan_test_A = experimento_ganancia(tree_params_A, X, y, semilla=semillas[0])
gan_test_B = experimento_ganancia(tree_params_B, X, y, semilla=semillas[0])

In [ ]:
df_comparacion = pd.DataFrame({
    'Ganancia': gan_test_A + gan_test_B,
    'Configuración': ['A: ' + str(tree_params_A)] * len(gan_test_A)
                    + ['B: ' + str(tree_params_B)] * len(gan_test_B)
})

plt.figure(figsize=(10, 6))
sns.histplot(data=df_comparacion, x='Ganancia', hue='Configuración', kde=True,
             element='step', stat='density', common_norm=False)
plt.title('Comparación de la distribución de ganancia (test) entre 2 configuraciones')
plt.tight_layout()
plt.show()

df_comparacion.groupby('Configuración')['Ganancia'].describe()